# Retail Strategy and Analytics - Task 2

In this notebook I'm trying to work out whether a store trial actually increased chip sales, by comparing each trial store to a similar "control" store that didn't run the trial.

In [ ]:
# Import the libraries I need
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from scipy import stats

In [ ]:
df = pd.read_csv('QVI_data.csv')
df.head()

The client picked stores 77, 86 and 88 as trial stores, with a trial period running from February to April 2019. Control stores need to be stores that were open the whole time we have data for.

To pick good control stores, I want to match on three things, measured before the trial started (i.e. before Feb 2019):
- Monthly total sales revenue
- Monthly number of customers
- Monthly number of transactions per customer

First step: build these metrics, and add a column with just the year and month of each transaction.

In [ ]:
# Turn DATE into an actual datetime, then pull out a YEARMONTH column
df['DATE'] = pd.to_datetime(df['DATE'])
df['YEARMONTH'] = df['DATE'].dt.strftime('%Y%m').astype('int64')
df.head()

Now I'll write some code to calculate, for every store and month: total sales, number of customers, transactions per customer, chips per transaction, and average price per unit.

In [ ]:
# Group the data by store and month so I can calculate metrics for each combination
grouped_df = df.groupby(['STORE_NBR', 'YEARMONTH'])

tot_sales = grouped_df['TOT_SALES'].sum()
n_cust = grouped_df['LYLTY_CARD_NBR'].nunique()
ntrans_percust = grouped_df['TXN_ID'].size() / n_cust
nchips_pertrans = grouped_df['PROD_QTY'].sum() / grouped_df['TXN_ID'].size()
avg_priceperunit = tot_sales / grouped_df['PROD_QTY'].sum()

# Put all the metrics together into one table
metrics_df = pd.concat([tot_sales, n_cust, ntrans_percust, nchips_pertrans, avg_priceperunit], axis=1)
metrics_df.columns = ['tot_sales', 'n_cust', 'ntrans_percust', 'nchips_pertrans', 'avg_priceperunit']
metrics_df = metrics_df.reset_index()
metrics_df.head()

In [ ]:
# I only want stores that have a full 12 months of data
month_counts = metrics_df.groupby('STORE_NBR')['YEARMONTH'].nunique().reset_index()
stores_fullobs = month_counts[month_counts['YEARMONTH'] == 12]['STORE_NBR']

# And I only want to compare on the months before the trial started
pretrial_metrics = metrics_df[metrics_df['STORE_NBR'].isin(stores_fullobs)]
pretrial_metrics = pretrial_metrics[pretrial_metrics['YEARMONTH'] < 201902]
pretrial_metrics.head()

Now I need a way to rank how similar each possible control store is to a given trial store. I'll do this two ways: correlation between the stores' metrics, and how close together the actual metric values are (magnitude distance).

In [ ]:
# Work out the correlation between a trial store and every possible control store.
# trial: the trial store number
# metric_col: list of metric column name(s) to compare on
def calc_corr(trial, metric_col):
    trial_stores = [77, 86, 88]
    # Every store with full data, except the trial stores themselves, is a possible control
    control_stores = stores_fullobs[~stores_fullobs.isin(trial_stores)]

    trial_vals = pretrial_metrics[pretrial_metrics['STORE_NBR'] == trial][metric_col].reset_index()

    corr_table = pd.DataFrame(columns=['YEARMONTH', 'trial_store', 'control_store', 'correlation'])

    for control in control_stores:
        control_vals = pretrial_metrics[pretrial_metrics['STORE_NBR'] == control][metric_col].reset_index()

        corr_row = pd.DataFrame(columns=['YEARMONTH', 'trial_store', 'control_store', 'correlation'])
        corr_row['YEARMONTH'] = list(pretrial_metrics.loc[pretrial_metrics['STORE_NBR'] == control, 'YEARMONTH'])
        corr_row['trial_store'] = trial
        corr_row['control_store'] = control
        # Correlate the trial store and control store's metric values, month by month
        corr_row['correlation'] = control_vals.corrwith(trial_vals, axis=1)

        corr_table = pd.concat([corr_table, corr_row])

    return corr_table


In [ ]:
trial_stores = [77, 86, 88]
corr_table = pd.DataFrame(columns=['YEARMONTH', 'trial_store', 'control_store', 'correlation'])

for store in trial_stores:
    corr_section = calc_corr(store, ['tot_sales', 'n_cust', 'ntrans_percust', 'nchips_pertrans', 'avg_priceperunit'])
    corr_table = pd.concat([corr_table, corr_section])

In [ ]:
corr_table.head()

As well as correlation, I'll also work out a standardised distance measure based on how far apart the trial store's and control store's actual values are.

In [ ]:
# Work out a normalised distance between a trial store and every possible control store.
# Same inputs as calc_corr above.
def calc_magdist(trial, metric_col):
    trial_stores = [77, 86, 88]
    control_stores = stores_fullobs[~stores_fullobs.isin(trial_stores)]

    dist_table = pd.DataFrame()

    for control in control_stores:
        trial_rows = pretrial_metrics[pretrial_metrics['STORE_NBR'] == trial].reset_index()[metric_col]
        control_rows = pretrial_metrics[pretrial_metrics['STORE_NBR'] == control].reset_index()[metric_col]

        # Distance is just the absolute difference between the two stores' values
        dist_row = abs(trial_rows - control_rows)
        dist_row.insert(0, 'YEARMONTH', list(pretrial_metrics.loc[pretrial_metrics['STORE_NBR'] == trial, 'YEARMONTH']))
        dist_row.insert(1, 'trial_store', trial)
        dist_row.insert(2, 'control_store', control)

        dist_table = pd.concat([dist_table, dist_row])

    # Scale each metric's distance to between 0 and 1, then flip it so 1 = most similar
    for col in metric_col:
        maxdist = dist_table[col].max()
        mindist = dist_table[col].min()
        dist_table[col] = 1 - (dist_table[col] - mindist) / (maxdist - mindist)

    # Average across metrics to get one overall similarity score per store/month
    dist_table['mag_measure'] = dist_table[metric_col].mean(axis=1)

    return dist_table


Now I'll use both functions together to score each possible control store, using total sales and number of customers - the two metrics the client cares most about - and average correlation/distance equally to get one final score.


In [ ]:
# For each trial store, score every possible control store using tot_sales and n_cust,
# then keep the top 5 control stores by combined score
trial_stores = [77, 86, 88]
metric_cols = ['tot_sales', 'n_cust']
corr_weight = 0.5  # weight correlation and magnitude distance equally

for trial in trial_stores:
    print('Trial store:', trial)

    scores = None
    for metric in metric_cols:
        corr_vals = calc_corr(trial, [metric])
        mag_vals = calc_magdist(trial, [metric]).drop(columns=[metric])  # avoid duplicate columns when merging
        combined = pd.merge(corr_vals, mag_vals, on=['YEARMONTH', 'trial_store', 'control_store'])

        # Average over all the pre-trial months, then combine correlation and distance into one score
        avg_scores = combined.groupby(['trial_store', 'control_store']).mean().reset_index()
        avg_scores[metric + '_score'] = corr_weight * avg_scores['correlation'] + (1 - corr_weight) * avg_scores['mag_measure']
        avg_scores = avg_scores[['control_store', metric + '_score']]

        scores = avg_scores if scores is None else scores.merge(avg_scores, on='control_store')

    # Give tot_sales and n_cust equal weight in the final score
    scores['final_score'] = 0.5 * scores['tot_sales_score'] + 0.5 * scores['n_cust_score']
    print(scores.sort_values(by='final_score', ascending=False).reset_index(drop=True).head(5))
    print()


Based on this, the best matching control store for each trial store is:

- Store 233 for trial store 77
- Store 155 for trial store 86
- Store 40 for trial store 88

Store 88's best score is noticeably lower than 77's or 86's, so its control store might not be as good a match.

Now that I have the control stores, let's visually check the trends look similar in the pre-trial period.

In [ ]:
def make_plots(storepair, metric_col):
    trial, control = storepair

    trial_plot = pretrial_metrics[pretrial_metrics['STORE_NBR'] == trial][['YEARMONTH', 'STORE_NBR', metric_col]]
    trial_plot = trial_plot.rename(columns={metric_col: metric_col + '_trial'})

    control_plot = pretrial_metrics[pretrial_metrics['STORE_NBR'] == control][['YEARMONTH', 'STORE_NBR', metric_col]]
    control_plot = control_plot.rename(columns={metric_col: metric_col + '_control'})

    # Also show the average of all the other stores, as a general reference line
    other_stores = pretrial_metrics[(pretrial_metrics['STORE_NBR'] != trial) & (pretrial_metrics['STORE_NBR'] != control)]
    plot_other = other_stores.groupby('YEARMONTH')[metric_col].mean()

    ax = control_plot.plot.line(x='YEARMONTH', y=metric_col + '_control', use_index=False, label='Control ' + metric_col)
    trial_plot.plot.line(x='YEARMONTH', y=metric_col + '_trial', use_index=False, ax=ax, label='Trial ' + metric_col)
    plot_other.plot.line(use_index=False, ax=ax, label='Other ' + metric_col)

    ax.set_ylabel(metric_col)
    plt.legend(title='STORE_NBR', loc='upper left', bbox_to_anchor=(1.0, 1.0))

    positions = range(7)
    labels = ('201807', '201808', '201809', '201810', '201811', '201812', '201901')
    plt.xticks(positions, labels)

    titlestr = 'The Trial Store ' + str(trial) + ' and Control Store ' + str(control) + ' in the Pre-Trial Period'
    ax.set_title(titlestr)
    plt.show()

In [ ]:
storepair = [[77, 233], [86, 155], [88, 40]]
metric_col = ['tot_sales', 'n_cust']

for pair in storepair:
    for metric in metric_col:
        make_plots(pair, metric)

The trial and control stores look reasonably similar in the pre-trial period, which is a good sign.

Now let's actually check whether chip sales went up during the trial. I'll start by scaling each control store's sales so it's on the same level as its trial store, to account for the two stores just being different sizes.

In [ ]:
# Work out a scaling factor for each store pair, based on total pre-trial sales
scale_store77 = pretrial_metrics[pretrial_metrics['STORE_NBR'] == 77]['tot_sales'].sum() / pretrial_metrics[pretrial_metrics['STORE_NBR'] == 233]['tot_sales'].sum()
scale_store86 = pretrial_metrics[pretrial_metrics['STORE_NBR'] == 86]['tot_sales'].sum() / pretrial_metrics[pretrial_metrics['STORE_NBR'] == 155]['tot_sales'].sum()
scale_store88 = pretrial_metrics[pretrial_metrics['STORE_NBR'] == 88]['tot_sales'].sum() / pretrial_metrics[pretrial_metrics['STORE_NBR'] == 40]['tot_sales'].sum()

In [ ]:
# Scale each control store's sales up (or down) using its scaling factor
scaled_control233 = metrics_df[metrics_df['STORE_NBR'] == 233][['STORE_NBR', 'YEARMONTH', 'tot_sales']].copy()
scaled_control233['tot_sales'] *= scale_store77

scaled_control155 = metrics_df[metrics_df['STORE_NBR'] == 155][['STORE_NBR', 'YEARMONTH', 'tot_sales']].copy()
scaled_control155['tot_sales'] *= scale_store86

scaled_control40 = metrics_df[metrics_df['STORE_NBR'] == 40][['STORE_NBR', 'YEARMONTH', 'tot_sales']].copy()
scaled_control40['tot_sales'] *= scale_store88

# Put all three scaled control stores into one table
scaledsales_control = pd.concat([scaled_control233, scaled_control155, scaled_control40]).reset_index(drop=True)
scaledsales_control = scaledsales_control.rename(columns={'tot_sales': 'scaled_tot_sales', 'STORE_NBR': 'CONTROL_NBR'})

# Also pull out the trial stores' own sales
trialsales = metrics_df[metrics_df['STORE_NBR'].isin([77, 86, 88])][['STORE_NBR', 'YEARMONTH', 'tot_sales']].reset_index(drop=True)
trialsales = trialsales.rename(columns={'STORE_NBR': 'TRIAL_NBR'})

Now that the control store sales are on a comparable scale, I can work out the percentage difference between the (scaled) control store and the trial store for each month.

In [ ]:
# Calculate the percentage difference between the trial and (scaled) control store, month by month
percentdiff = scaledsales_control.copy()
percentdiff[['TRIAL_NBR', 'tot_sales_t']] = trialsales[['TRIAL_NBR', 'tot_sales']]
percentdiff = percentdiff.rename(columns={'scaled_tot_sales': 'scaled_sales_c'})

percentdiff['sales_percent_diff'] = (percentdiff['tot_sales_t'] - percentdiff['scaled_sales_c']) / \
                                     (0.5 * (percentdiff['scaled_sales_c'] + percentdiff['tot_sales_t']))
percentdiff.head()

Let's check if this difference is actually statistically significant using a t-test. The null hypothesis is that the trial period is no different to the pre-trial period, i.e. a 0% difference between trial and control.

In [ ]:
# Use the standard deviation of the percentage difference in the pre-trial period as our baseline
storepair = [[77, 233], [86, 155], [88, 40]]

pretrial_percentdiff = percentdiff[percentdiff['YEARMONTH'] < 201902]
dof = 6  # 7 months of pre-trial data, minus 1

for trialstore, controlstore in storepair:
    pretrial = pretrial_percentdiff[pretrial_percentdiff['TRIAL_NBR'] == trialstore]
    std = pretrial['sales_percent_diff'].std()
    mean = pretrial['sales_percent_diff'].mean()

    trialperiod = percentdiff[(percentdiff['YEARMONTH'] >= 201902) & (percentdiff['YEARMONTH'] <= 201904) &
                               (percentdiff['TRIAL_NBR'] == trialstore)]

    print('Trial store -', trialstore, '; control store -', controlstore)
    print('Month : t-statistic')
    for month in trialperiod['YEARMONTH'].unique():
        xval = trialperiod[trialperiod['YEARMONTH'] == month]['sales_percent_diff'].item()
        tstat = (xval - mean) / std
        print(str(month), ':', tstat)
    print()

# The t-statistic for the 95th percentile with 6 degrees of freedom
print('95th percentile value:', stats.t.ppf(1 - 0.05, dof))

The t-value for trial store 77 is well above the 95th percentile threshold in both March and April - so the increase in sales there is statistically significant. The same is true for store 86 in March.

Let's make this easier to see by plotting the trial and control sales against the control store's 5th-95th percentile range.

In [ ]:
# Bar charts focused on just the trial period
storepair = [[77, 233], [86, 155], [88, 40]]

for trial, control in storepair:
    plot_control = percentdiff[(percentdiff['CONTROL_NBR'] == control) & (percentdiff['YEARMONTH'] >= 201902) & (percentdiff['YEARMONTH'] <= 201904)][['YEARMONTH', 'CONTROL_NBR', 'scaled_sales_c']]
    plot_control = plot_control.rename(columns={'CONTROL_NBR': 'STORE_NBR', 'scaled_sales_c': 'control_sales'})

    plot_trial = percentdiff[(percentdiff['TRIAL_NBR'] == trial) & (percentdiff['YEARMONTH'] >= 201902) & (percentdiff['YEARMONTH'] <= 201904)][['YEARMONTH', 'TRIAL_NBR', 'tot_sales_t']]
    plot_trial = plot_trial.rename(columns={'TRIAL_NBR': 'STORE_NBR', 'tot_sales_t': 'trial_sales'})

    toplot = plot_control[['YEARMONTH', 'control_sales']].merge(plot_trial[['YEARMONTH', 'trial_sales']], on='YEARMONTH').set_index('YEARMONTH')
    ax = toplot.plot(kind='bar', figsize=(7, 5))

    # Add lines for the control store's 5th and 95th percentile range, based on the pre-trial standard deviation
    std = percentdiff[(percentdiff['CONTROL_NBR'] == control) & (percentdiff['YEARMONTH'] < 201902)]['sales_percent_diff'].std()

    threshold95 = plot_control.reset_index()[['YEARMONTH', 'control_sales']].copy()
    threshold95['control_sales'] = threshold95['control_sales'] * (1 + std * 2)

    threshold5 = plot_control.reset_index()[['YEARMONTH', 'control_sales']].copy()
    threshold5['control_sales'] = threshold5['control_sales'] * (1 - std * 2)

    threshold95.plot.line(x='YEARMONTH', y='control_sales', color='y', figsize=(7, 5), use_index=False, ax=ax)
    threshold5.plot.line(x='YEARMONTH', y='control_sales', color='g', figsize=(7, 5), use_index=False, ax=ax)

    plt.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
    titlestr = 'Trial Store ' + str(trial) + ' and Control Store ' + str(control) + ' in the Trial Period'
    ax.set_title(titlestr)
    plt.show()

In [ ]:
# Line graphs over the full year, for the report
storepair = [[77, 233], [86, 155], [88, 40]]

for trial, control in storepair:
    plot_control = percentdiff[percentdiff['CONTROL_NBR'] == control][['YEARMONTH', 'CONTROL_NBR', 'scaled_sales_c']]
    plot_control = plot_control.rename(columns={'CONTROL_NBR': 'STORE_NBR', 'scaled_sales_c': 'control_sales'})

    plot_trial = percentdiff[percentdiff['TRIAL_NBR'] == trial][['YEARMONTH', 'TRIAL_NBR', 'tot_sales_t']]
    plot_trial = plot_trial.rename(columns={'TRIAL_NBR': 'STORE_NBR', 'tot_sales_t': 'trial_sales'})

    ax = plot_control.plot.line(x='YEARMONTH', y='control_sales', use_index=False, label='Control Sales')
    plot_trial.plot.line(x='YEARMONTH', y='trial_sales', use_index=False, ax=ax, label='Trial Sales')

    std = percentdiff[(percentdiff['CONTROL_NBR'] == control) & (percentdiff['YEARMONTH'] < 201902)]['sales_percent_diff'].std()

    threshold95 = plot_control.reset_index()[['YEARMONTH', 'control_sales']].copy()
    threshold95['control_sales'] = threshold95['control_sales'] * (1 + std * 2)

    threshold5 = plot_control.reset_index()[['YEARMONTH', 'control_sales']].copy()
    threshold5['control_sales'] = threshold5['control_sales'] * (1 - std * 2)

    threshold95.plot.line(x='YEARMONTH', y='control_sales', color='y', linestyle='--', figsize=(10, 5), use_index=False, ax=ax, label='95th Percentile')
    threshold5.plot.line(x='YEARMONTH', y='control_sales', color='g', linestyle='--', figsize=(10, 5), use_index=False, ax=ax, label='5th Percentile')

    # Shade the trial period so it's easy to spot on the chart
    plt.axvspan(6.5, 9.5, alpha=0.2, label='Trial period')

    ax.set_ylabel('Total Sales')
    plt.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
    titlestr = 'Performance of Trial Store ' + str(trial) + ' and Control Store ' + str(control) + ' Over the 2018/19 Year'

    positions = range(12)
    labels = ('201807', '201808', '201809', '201810', '201811', '201812', '201901', '201902', '201903', '201904', '201905', '201906')
    plt.xticks(positions, labels)

    ax.set_title(titlestr)
    plt.show()

Store 77's trial period sales fall outside the control store's 5th-95th percentile range in two of the three trial months, so the difference there looks genuinely significant.

For store 86, March is significantly different from the control store, but February and April aren't.

Store 88 shows no significant difference from its control store during the trial.

Let's check the same thing for number of customers.

In [ ]:
# Same scaling approach as before, but for number of customers instead of sales
scale_store77 = pretrial_metrics[pretrial_metrics['STORE_NBR'] == 77]['n_cust'].sum() / pretrial_metrics[pretrial_metrics['STORE_NBR'] == 233]['n_cust'].sum()
scale_store86 = pretrial_metrics[pretrial_metrics['STORE_NBR'] == 86]['n_cust'].sum() / pretrial_metrics[pretrial_metrics['STORE_NBR'] == 155]['n_cust'].sum()
scale_store88 = pretrial_metrics[pretrial_metrics['STORE_NBR'] == 88]['n_cust'].sum() / pretrial_metrics[pretrial_metrics['STORE_NBR'] == 40]['n_cust'].sum()

In [ ]:
scaled_control233 = metrics_df[metrics_df['STORE_NBR'] == 233][['STORE_NBR', 'YEARMONTH', 'n_cust']].copy()
scaled_control233['n_cust'] *= scale_store77

scaled_control155 = metrics_df[metrics_df['STORE_NBR'] == 155][['STORE_NBR', 'YEARMONTH', 'n_cust']].copy()
scaled_control155['n_cust'] *= scale_store86

scaled_control40 = metrics_df[metrics_df['STORE_NBR'] == 40][['STORE_NBR', 'YEARMONTH', 'n_cust']].copy()
scaled_control40['n_cust'] *= scale_store88

scaledncust_control = pd.concat([scaled_control233, scaled_control155, scaled_control40]).reset_index(drop=True)
scaledncust_control = scaledncust_control.rename(columns={'n_cust': 'scaled_n_cust', 'STORE_NBR': 'CONTROL_NBR'})

trialncust = metrics_df[metrics_df['STORE_NBR'].isin([77, 86, 88])][['STORE_NBR', 'YEARMONTH', 'n_cust']].reset_index(drop=True)
trialncust = trialncust.rename(columns={'STORE_NBR': 'TRIAL_NBR'})

In [ ]:
percentdiff = scaledncust_control.copy()
percentdiff[['TRIAL_NBR', 'n_cust_t']] = trialncust[['TRIAL_NBR', 'n_cust']]
percentdiff = percentdiff.rename(columns={'scaled_n_cust': 'scaled_n_cust_c'})

percentdiff['cust_percent_diff'] = (percentdiff['n_cust_t'] - percentdiff['scaled_n_cust_c']) / \
                                    (0.5 * (percentdiff['scaled_n_cust_c'] + percentdiff['n_cust_t']))
percentdiff.head()

In [ ]:
storepair = [[77, 233], [86, 155], [88, 40]]

pretrial_percentdiff = percentdiff[percentdiff['YEARMONTH'] < 201902]
dof = 6

for trialstore, controlstore in storepair:
    pretrial = pretrial_percentdiff[pretrial_percentdiff['TRIAL_NBR'] == trialstore]
    std = pretrial['cust_percent_diff'].std()
    mean = pretrial['cust_percent_diff'].mean()

    trialperiod = percentdiff[(percentdiff['YEARMONTH'] >= 201902) & (percentdiff['YEARMONTH'] <= 201904) &
                               (percentdiff['TRIAL_NBR'] == trialstore)]

    print('Trial store -', trialstore, '; control store -', controlstore)
    print('Month : t-statistic')
    for month in trialperiod['YEARMONTH'].unique():
        xval = trialperiod[trialperiod['YEARMONTH'] == month]['cust_percent_diff'].item()
        tstat = (xval - mean) / std
        print(str(month), ':', tstat)
    print()

print('95th percentile value:', stats.t.ppf(1 - 0.05, dof))

Similar to total sales, there are statistically significant increases in the number of customers for stores 77 and 86 in at least two of the trial months. Store 88 shows no significant increase.

Let's visualise this the same way as before.

In [ ]:
# Bar charts focused on the trial period
storepair = [[77, 233], [86, 155], [88, 40]]

for trial, control in storepair:
    plot_control = percentdiff[(percentdiff['CONTROL_NBR'] == control) & (percentdiff['YEARMONTH'] >= 201902) & (percentdiff['YEARMONTH'] <= 201904)][['YEARMONTH', 'CONTROL_NBR', 'scaled_n_cust_c']]
    plot_control = plot_control.rename(columns={'CONTROL_NBR': 'STORE_NBR', 'scaled_n_cust_c': 'control_ncust'})

    plot_trial = percentdiff[(percentdiff['TRIAL_NBR'] == trial) & (percentdiff['YEARMONTH'] >= 201902) & (percentdiff['YEARMONTH'] <= 201904)][['YEARMONTH', 'TRIAL_NBR', 'n_cust_t']]
    plot_trial = plot_trial.rename(columns={'TRIAL_NBR': 'STORE_NBR', 'n_cust_t': 'trial_ncust'})

    toplot = plot_control[['YEARMONTH', 'control_ncust']].merge(plot_trial[['YEARMONTH', 'trial_ncust']], on='YEARMONTH').set_index('YEARMONTH')
    ax = toplot.plot(kind='bar', figsize=(7, 5))

    std = percentdiff[(percentdiff['CONTROL_NBR'] == control) & (percentdiff['YEARMONTH'] < 201902)]['cust_percent_diff'].std()

    threshold95 = plot_control.reset_index()[['YEARMONTH', 'control_ncust']].copy()
    threshold95['control_ncust'] = threshold95['control_ncust'] * (1 + std * 2)

    threshold5 = plot_control.reset_index()[['YEARMONTH', 'control_ncust']].copy()
    threshold5['control_ncust'] = threshold5['control_ncust'] * (1 - std * 2)

    threshold95.plot.line(x='YEARMONTH', y='control_ncust', color='y', figsize=(7, 5), use_index=False, ax=ax)
    threshold5.plot.line(x='YEARMONTH', y='control_ncust', color='g', figsize=(7, 5), use_index=False, ax=ax)

    plt.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
    titlestr = 'Trial Store ' + str(trial) + ' and Control Store ' + str(control) + ' in the Trial Period'
    ax.set_title(titlestr)
    plt.show()

In [ ]:
# Line graphs over the full year
storepair = [[77, 233], [86, 155], [88, 40]]

for trial, control in storepair:
    plot_control = percentdiff[percentdiff['CONTROL_NBR'] == control][['YEARMONTH', 'CONTROL_NBR', 'scaled_n_cust_c']]
    plot_control = plot_control.rename(columns={'CONTROL_NBR': 'STORE_NBR', 'scaled_n_cust_c': 'control_ncust'})

    plot_trial = percentdiff[percentdiff['TRIAL_NBR'] == trial][['YEARMONTH', 'TRIAL_NBR', 'n_cust_t']]
    plot_trial = plot_trial.rename(columns={'TRIAL_NBR': 'STORE_NBR', 'n_cust_t': 'trial_ncust'})

    ax = plot_control.plot.line(x='YEARMONTH', y='control_ncust', use_index=False, label='Control No. Cust')
    plot_trial.plot.line(x='YEARMONTH', y='trial_ncust', use_index=False, ax=ax, label='Trial No. Cust')

    std = percentdiff[(percentdiff['CONTROL_NBR'] == control) & (percentdiff['YEARMONTH'] < 201902)]['cust_percent_diff'].std()

    threshold95 = plot_control.reset_index()[['YEARMONTH', 'control_ncust']].copy()
    threshold95['control_ncust'] = threshold95['control_ncust'] * (1 + std * 2)

    threshold5 = plot_control.reset_index()[['YEARMONTH', 'control_ncust']].copy()
    threshold5['control_ncust'] = threshold5['control_ncust'] * (1 - std * 2)

    threshold95.plot.line(x='YEARMONTH', y='control_ncust', color='y', linestyle='--', figsize=(10, 5), use_index=False, ax=ax, label='95th Percentile')
    threshold5.plot.line(x='YEARMONTH', y='control_ncust', color='g', linestyle='--', figsize=(10, 5), use_index=False, ax=ax, label='5th Percentile')

    plt.axvspan(6.5, 9.5, alpha=0.2, label='Trial period')

    ax.set_ylabel('No. Customers')
    plt.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
    titlestr = 'Performance of Trial Store ' + str(trial) + ' and Control Store ' + str(control) + ' Over the 2018/19 Year'

    positions = range(12)
    labels = ('201807', '201808', '201809', '201810', '201811', '201812', '201901', '201902', '201903', '201904', '201905', '201906')
    plt.xticks(positions, labels)

    ax.set_title(titlestr)
    plt.show()

Customer numbers are significantly higher in all three trial months for stores 77 and 86. Store 86's customer increase looks stronger than its sales increase, which might be worth asking the Category Manager about - maybe there were special deals in the trial store that brought in more customers but at lower prices. As with total sales, store 88 shows no significant difference from its control store.

## Conclusions

Trial stores 77 and 86 both show a statistically significant difference from their control stores in at least two of the three trial months, for both sales and customer numbers. Store 88 doesn't show a significant difference, which might mean the trial was run differently there, or that it just wasn't as effective in that store. Overall though, the trial does appear to have led to a real increase in sales in two out of the three trial stores.